In [1]:
import pandas as pd
import numpy as np
import torch
import transformers
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report, confusion_matrix
from transformers import AutoModel, BertTokenizerFast, DistilBertTokenizerFast
from torch.utils.data import TensorDataset, DataLoader, RandomSampler, SequentialSampler
import torch.nn as nn
from transformers import AdamW
from sklearn.utils.class_weight import compute_class_weight
from tqdm.notebook import tqdm
import os
import shap

In [2]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [3]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# Read Degendered Data

In [19]:
os.chdir('/content/drive/My Drive/NLP-Letters-V2/notebooks/')
degendered1 = pd.read_csv('../data/letters_2021_processed.csv')
degendered1 = degendered1[['s1_s2', 'full_text', 'LETTER_GENDER']]
degendered1 = degendered1.rename(columns={'LETTER_GENDER':'label'})

In [20]:
degendered2 = pd.read_csv('../data/sentence_sets_trimmed_processed.csv')
degendered2 = degendered2[['s1_s2', 'full_text', 'applicant_gender']]
degendered2 = degendered2.rename(columns={'applicant_gender':'label'})

In [21]:
degendered = pd.concat([degendered1, degendered2], ignore_index=True)

In [22]:
gender_label_mapping = {
    'F':0,
    'female':0,
    'M':1,
    'male':1
}

In [23]:
degendered['label'] = degendered['label'].replace(gender_label_mapping)

<ipython-input-23-b761b83cae52>:1: FutureWarning: Downcasting behavior in `replace` is deprecated and will be removed in a future version. To retain the old behavior, explicitly call `result.infer_objects(copy=False)`. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  degendered['label'] = degendered['label'].replace(gender_label_mapping)


# Read Gendered Data

In [24]:
gendered1 = pd.read_csv('../data/letters_2021_processed_with_gender.csv')
gendered1 = gendered1[['s1_s2', 'LETTER_GENDER']]
gendered1 = gendered1.rename(columns={'LETTER_GENDER':'label'})

In [25]:
gendered2 = pd.read_csv('../data/sentence_sets_trimmed_processed_with_gender.csv', encoding='mac-roman')
gendered2 = gendered2[['s1_s2', 'applicant_gender']]
gendered2 = gendered2.rename(columns={'TEXT':'LETTERTEXT', 'applicant_gender':'label'})

In [26]:
gendered = pd.concat([gendered1, gendered2], ignore_index=True)

In [27]:
gendered['label'] = gendered['label'].replace(gender_label_mapping)

<ipython-input-27-d6a02b119c3b>:1: FutureWarning: Downcasting behavior in `replace` is deprecated and will be removed in a future version. To retain the old behavior, explicitly call `result.infer_objects(copy=False)`. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  gendered['label'] = gendered['label'].replace(gender_label_mapping)


# Create Training and Test Sets

In [28]:
train_indices, test_indices = train_test_split(
    gendered.index,
    test_size=0.2,
    stratify=degendered['label'],
    random_state=42
)

In [29]:
gendered = gendered.loc[train_indices]

In [30]:
degendered = degendered.loc[test_indices]

In [31]:
train_text, val_text, train_labels, val_labels = train_test_split(gendered['s1_s2'], gendered['label'],
                                                                    random_state=0,
                                                                    test_size=0.3,
                                                                    stratify=gendered['label'])


test_text = degendered['s1_s2']
test_labels = degendered['label']

In [32]:
bert = AutoModel.from_pretrained('distilbert-base-uncased')
tokenizer = DistilBertTokenizerFast.from_pretrained('distilbert-base-uncased')

model.safetensors:   0%|          | 0.00/268M [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/48.0 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

In [33]:
tokens_train = tokenizer.batch_encode_plus(
    train_text.tolist(),
    padding='max_length',
    truncation=True
)

tokens_val = tokenizer.batch_encode_plus(
    val_text.tolist(),
    padding='max_length',
    truncation=True
)

tokens_test = tokenizer.batch_encode_plus(
    test_text.tolist(),
    padding='max_length',
    truncation=True
)

In [34]:
train_seq = torch.tensor(tokens_train['input_ids'])
train_mask = torch.tensor(tokens_train['attention_mask'])
train_y = torch.tensor(train_labels.tolist())

val_seq = torch.tensor(tokens_val['input_ids'])
val_mask = torch.tensor(tokens_val['attention_mask'])
val_y = torch.tensor(val_labels.tolist())

test_seq = torch.tensor(tokens_test['input_ids'])
test_mask = torch.tensor(tokens_test['attention_mask'])
test_y = torch.tensor(test_labels.tolist())

In [35]:
batch_size = 8
train_data = TensorDataset(train_seq, train_mask, train_y)
train_sampler = RandomSampler(train_data)
train_dataloader = DataLoader(train_data, sampler=train_sampler, batch_size=batch_size)

val_data = TensorDataset(val_seq, val_mask, val_y)
val_sampler = SequentialSampler(val_data)
val_dataloader = DataLoader(val_data, sampler = val_sampler, batch_size=batch_size)

In [36]:
# freeze weights, uncomment if you would like to unfreeze weights
for param in bert.parameters():
    param.requires_grad = False


# Create Model

In [37]:
class BERT_Arch(nn.Module):

    def __init__(self, bert):

        super(BERT_Arch, self).__init__()

        self.bert = bert

        # dropout layer
        self.dropout = nn.Dropout(0.1)

        # relu activation function
        self.relu =  nn.ReLU()

        # dense layer 1
        self.fc1 = nn.Linear(768,512)

        # dense layer 2 (Output layer)
        self.fc2 = nn.Linear(512,2)

        #softmax activation function
        self.softmax = nn.LogSoftmax(dim=1)

    #define the forward pass
    def forward(self, sent_id, mask):

        #pass the inputs to the model
        distilbert_output = self.bert(sent_id, attention_mask=mask)

        hidden_state = distilbert_output[0]  # (bs, seq_len, dim)
        pooled_output = hidden_state[:, 0]
        # pooled_output = hidden_state.mean(dim=1)

        x = self.fc1(pooled_output)

        x = self.relu(x)

        x = self.dropout(x)

        # output layer
        x = self.fc2(x)

        # apply softmax activation
        x = self.softmax(x)

        return x

In [38]:
model = BERT_Arch(bert)
model = model.to(device)

In [39]:
optimizer = torch.optim.Adam(model.parameters(), lr=1e-4)
class_weights = compute_class_weight(class_weight='balanced', classes=np.unique(train_labels), y=train_labels)
weights= torch.tensor(class_weights,dtype=torch.float)
weights = weights.to(device)
cross_entropy  = nn.NLLLoss(weight=weights)
epochs = 10

In [40]:
# function to train the model
def train():

  model.train()

  total_loss, total_accuracy = 0, 0

  # empty list to save model predictions
  total_preds=[]
  total_labels=[]

  # iterate over batches
  for step,batch in enumerate(tqdm(train_dataloader, desc="Training", leave=True)):

    # # progress update after every 50 batches.
    # if step % 50 == 0 and not step == 0:
    #   print('  Batch {:>5,}  of  {:>5,}.'.format(step, len(train_dataloader)))

    # push the batch to gpu
    batch = [r.to(device) for r in batch]

    sent_id, mask, labels = batch

    # clear previously calculated gradients
    model.zero_grad()

    # get model predictions for the current batch
    preds = model(sent_id, mask)

    # compute the loss between actual and predicted values
    loss = cross_entropy(preds, labels)

    # add on to the total loss
    total_loss = total_loss + loss.item()

    # backward pass to calculate the gradients
    loss.backward()

    # clip the the gradients to 1.0. It helps in preventing the exploding gradient problem
    torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)

    # update parameters
    optimizer.step()

    # model predictions are stored on GPU. So, push it to CPU
    preds=preds.detach().cpu().numpy()

    # append the model predictions
    total_preds.append(preds)

    labels=labels.detach().cpu().numpy()

    # append labels
    total_labels.append(labels)

  # compute the training loss of the epoch
  avg_loss = total_loss / len(train_dataloader)

  # predictions are in the form of (no. of batches, size of batch, no. of classes).
  # reshape the predictions in form of (number of samples, no. of classes)
  total_preds  = np.concatenate(total_preds, axis=0)

  total_labels = np.concatenate(total_labels, axis=0)

  epoch_preds = np.argmax(total_preds, axis = 1)

  print('Training Classification Report: ', classification_report(total_labels, epoch_preds))
  print('Training Confusion Matrix: ', confusion_matrix(total_labels, epoch_preds))

  #returns the loss and predictions
  return avg_loss, total_preds

In [41]:
# function for evaluating the model
def evaluate():

  print("\nEvaluating...")

  # deactivate dropout layers
  model.eval()

  total_loss, total_accuracy = 0, 0

  # empty list to save the model predictions
  total_preds = []
  total_labels = []

  # iterate over batches
  for step,batch in enumerate(val_dataloader):

    # # Progress update every 50 batches.
    # if step % 50 == 0 and not step == 0:

    #   # Calculate elapsed time in minutes.
    #   elapsed = format_time(time.time() - t0)

    #   # Report progress.
    #   print('  Batch {:>5,}  of  {:>5,}.'.format(step, len(val_dataloader)))

    # push the batch to gpu
    batch = [t.to(device) for t in batch]

    sent_id, mask, labels = batch

    # deactivate autograd
    with torch.no_grad():

      # model predictions
      preds = model(sent_id, mask)

      # compute the validation loss between actual and predicted values
      loss = cross_entropy(preds,labels)

      total_loss = total_loss + loss.item()

      preds = preds.detach().cpu().numpy()

      total_preds.append(preds)

      labels = labels.detach().cpu().numpy()

      total_labels.append(labels)

  # compute the validation loss of the epoch
  avg_loss = total_loss / len(val_dataloader)

  # reshape the predictions in form of (number of samples, no. of classes)
  total_preds  = np.concatenate(total_preds, axis=0)

  total_labels = np.concatenate(total_labels, axis=0)

  epoch_preds = np.argmax(total_preds, axis = 1)

  print('Validation Classification Report: ', classification_report(total_labels, epoch_preds))
  print('Validation Confusion Matrix: ', confusion_matrix(total_labels, epoch_preds))

  return avg_loss, total_preds

In [42]:
# set initial loss to infinite
best_valid_loss = float('inf')

# empty lists to store training and validation loss of each epoch
train_losses=[]
valid_losses=[]

#for each epoch
for epoch in tqdm(range(epochs), desc="Training"):

    print('\n Epoch {:} / {:}'.format(epoch + 1, epochs))

    #train model
    train_loss, _ = train()

    #evaluate model
    valid_loss, _ = evaluate()

    #save the best model
    if valid_loss < best_valid_loss:
        best_valid_loss = valid_loss
        print('Model Saved!')
        torch.save(model, '../saved_models/models_base/saved_model_degendered.pt')

    # append training and validation loss
    train_losses.append(train_loss)
    valid_losses.append(valid_loss)

    print(f'\nTraining Loss: {train_loss:.3f}')
    print(f'Validation Loss: {valid_loss:.3f}')

Training:   0%|          | 0/10 [00:00<?, ?it/s]


 Epoch 1 / 10


Training:   0%|          | 0/629 [00:00<?, ?it/s]

Training Classification Report:                precision    recall  f1-score   support

           0       1.00      0.97      0.99      1560
           1       0.99      1.00      0.99      3472

    accuracy                           0.99      5032
   macro avg       0.99      0.99      0.99      5032
weighted avg       0.99      0.99      0.99      5032

Training Confusion Matrix:  [[1519   41]
 [   2 3470]]

Evaluating...
Validation Classification Report:                precision    recall  f1-score   support

           0       1.00      1.00      1.00       669
           1       1.00      1.00      1.00      1488

    accuracy                           1.00      2157
   macro avg       1.00      1.00      1.00      2157
weighted avg       1.00      1.00      1.00      2157

Validation Confusion Matrix:  [[ 669    0]
 [   0 1488]]
Model Saved!

Training Loss: 0.074
Validation Loss: 0.002

 Epoch 2 / 10


Training:   0%|          | 0/629 [00:00<?, ?it/s]

...

Training Classification Report:                precision    recall  f1-score   support

           0       1.00      1.00      1.00      1560
           1       1.00      1.00      1.00      3472

    accuracy                           1.00      5032
   macro avg       1.00      1.00      1.00      5032
weighted avg       1.00      1.00      1.00      5032

Training Confusion Matrix:  [[1558    2]
 [   2 3470]]

Evaluating...
Validation Classification Report:                precision    recall  f1-score   support

           0       1.00      1.00      1.00       669
           1       1.00      1.00      1.00      1488

    accuracy                           1.00      2157
   macro avg       1.00      1.00      1.00      2157
weighted avg       1.00      1.00      1.00      2157

Validation Confusion Matrix:  [[ 669    0]
 [   0 1488]]
Model Saved!

Training Loss: 0.002
Validation Loss: 0.000

 Epoch 9 / 10


Training:   0%|          | 0/629 [00:00<?, ?it/s]

Training Classification Report:                precision    recall  f1-score   support

           0       1.00      1.00      1.00      1560
           1       1.00      1.00      1.00      3472

    accuracy                           1.00      5032
   macro avg       1.00      1.00      1.00      5032
weighted avg       1.00      1.00      1.00      5032

Training Confusion Matrix:  [[1559    1]
 [   2 3470]]

Evaluating...
Validation Classification Report:                precision    recall  f1-score   support

           0       1.00      1.00      1.00       669
           1       1.00      1.00      1.00      1488

    accuracy                           1.00      2157
   macro avg       1.00      1.00      1.00      2157
weighted avg       1.00      1.00      1.00      2157

Validation Confusion Matrix:  [[ 669    0]
 [   0 1488]]
Model Saved!

Training Loss: 0.002
Validation Loss: 0.000

 Epoch 10 / 10


Training:   0%|          | 0/629 [00:00<?, ?it/s]

Training Classification Report:                precision    recall  f1-score   support

           0       1.00      1.00      1.00      1560
           1       1.00      1.00      1.00      3472

    accuracy                           1.00      5032
   macro avg       1.00      1.00      1.00      5032
weighted avg       1.00      1.00      1.00      5032

Training Confusion Matrix:  [[1559    1]
 [   2 3470]]

Evaluating...
Validation Classification Report:                precision    recall  f1-score   support

           0       1.00      1.00      1.00       669
           1       1.00      1.00      1.00      1488

    accuracy                           1.00      2157
   macro avg       1.00      1.00      1.00      2157
weighted avg       1.00      1.00      1.00      2157

Validation Confusion Matrix:  [[ 669    0]
 [   1 1487]]

Training Loss: 0.001
Validation Loss: 0.001


# Apply on Degendered Test Set

In [43]:
model = torch.load("../saved_models/models_base/saved_model_degendered.pt")
model.eval()

<ipython-input-43-5c40f358b83b>:1: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  model = torch.load("../saved_models/models_base/saved_model_degendered.pt")


BERT_Arch(
  (bert): DistilBertModel(
    (embeddings): Embeddings(
      (word_embeddings): Embedding(30522, 768, padding_idx=0)
      (position_embeddings): Embedding(512, 768)
      (LayerNorm): LayerNorm((768,), eps=1e-12, elementwise_affine=True)
      (dropout): Dropout(p=0.1, inplace=False)
    )
    (transformer): Transformer(
      (layer): ModuleList(
        (0-5): 6 x TransformerBlock(
          (attention): DistilBertSdpaAttention(
            (dropout): Dropout(p=0.1, inplace=False)
            (q_lin): Linear(in_features=768, out_features=768, bias=True)
            (k_lin): Linear(in_features=768, out_features=768, bias=True)
            (v_lin): Linear(in_features=768, out_features=768, bias=True)
            (out_lin): Linear(in_features=768, out_features=768, bias=True)
          )
          (sa_layer_norm): LayerNorm((768,), eps=1e-12, elementwise_affine=True)
          (ffn): FFN(
            (dropout): Dropout(p=0.1, inplace=False)
            (lin1): Linear(in_fe

In [44]:
test_dataset = TensorDataset(test_seq, test_mask, test_y)

test_loader = DataLoader(test_dataset, batch_size=8, shuffle=False)

In [45]:
all_preds = []
all_labels = []

with torch.no_grad():  # Disable gradient calculations for efficiency
    for batch in tqdm(test_loader, desc="Processing Batches", unit="batch"):
        batch_seq, batch_mask, batch_labels = batch  # Extract input tensors
        batch_seq, batch_mask = batch_seq.to(device), batch_mask.to(device)  # Move to device

        outputs = model(batch_seq, mask=batch_mask)  # Forward pass
        preds = torch.argmax(outputs, dim=1)

        all_preds.extend(preds.cpu().numpy())  # Store predictions
        all_labels.extend(batch_labels.cpu().numpy())  # Store true labels


Processing Batches:   0%|          | 0/225 [00:00<?, ?batch/s]

In [46]:
  print('Test Classification Report: \n', classification_report(all_labels, all_preds))
  print('Test Confusion Matrix: \n', confusion_matrix(all_labels, all_preds))

Test Classification Report: 
               precision    recall  f1-score   support

           0       0.39      0.22      0.28       557
           1       0.71      0.84      0.77      1241

    accuracy                           0.65      1798
   macro avg       0.55      0.53      0.53      1798
weighted avg       0.61      0.65      0.62      1798

Test Confusion Matrix: 
 [[ 123  434]
 [ 194 1047]]
